# Week 6: Validation and Research Claim Audit

## Section 1: Research Paper Critique
We are auditing two core findings from the provided FlyRank research paper, evaluating the data labeling process and checking if the validation design safely supports the claims.

### Finding 1: AI-Generated Content Performance Scale
* **The Claim:** The paper states that AI-generated programmatic pages saw a direct traffic increase of over 40% across studied domains within 90 days.
* **Methodology Question:** How was the baseline domain authority and historical traffic growth controlled for before attributing the full 40% gain directly to the AI text generation? If the studied domains were already on an upward trend, the label suffers from confounding variables.

### Finding 2: Bounce Rate vs. Content Length
* **The Claim:** The study finds a strong negative correlation between long-form AI content lengths (2,000+ words) and user bounce rates.
* **Methodology Question:** Does the validation design account for user intent group-splitting? Long-form content typically satisfies informational queries where users naturally stay longer to read, whereas short-form answers transactional queries quickly (causing a safe "bounce"). Without grouping by search intent, this correlation might be a validation design artifact rather than a true content-length driver.s

In [7]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score

# 1. Ingest the raw data asset
relative_path = os.path.join("..", "..", "Data", "Raw", "PK_2017-18_DHS_06272026_421_252524", "PKHR71DT", "PKHR71FL.DTA")
df = pd.read_stata(relative_path, convert_categoricals=True)

# 2. Engineer the target variable honestly from the wealth index categories
df['target_stunting_risk'] = df['hv270'].isin(['poorest', 'poorer']).astype(int)

# 3. Define features (dropping the wealth index variable to avoid data leakage)
honest_features = ['hv001', 'hv024', 'hv025']
target_col = 'target_stunting_risk'

df_honest = df[honest_features + [target_col]].dropna().copy()
df_encoded = pd.get_dummies(df_honest, columns=['hv024', 'hv025'], drop_first=True)

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col].astype(int)
X_sm = sm.add_constant(X)

# 4. Re-run Week-5 Random Split (Before)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_sm, y, test_size=0.2, random_state=42, stratify=y
)
model_random = sm.Logit(y_train_r, X_train_r.astype(float)).fit(disp=0)
auc_random = roc_auc_score(y_test_r, model_random.predict(X_test_r.astype(float)))

# 5. Apply the Honest Group Split by Cluster (After)
# This prevents data leakage across neighboring households
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_sm, y, groups=df_honest['hv001']))

X_train_g, X_test_g = X_sm.iloc[train_idx], X_sm.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_group = sm.Logit(y_train_g, X_train_g.astype(float)).fit(disp=0)
auc_group = roc_auc_score(y_test_g, model_group.predict(X_test_g.astype(float)))

print("\n--- Validation Audit Split Comparison ---")
print(f"Random Split Validation ROC-AUC (Week 5): {auc_random:.4f}")
print(f"Group-Cluster Split Validation ROC-AUC (Week 6): {auc_group:.4f}")

C:\Users\Harris\AppData\Local\Temp\ipykernel_22216\3869421276.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target_stunting_risk'] = df['hv270'].isin(['poorest', 'poorer']).astype(int)



--- Validation Audit Split Comparison ---
Random Split Validation ROC-AUC (Week 5): 0.8316
Group-Cluster Split Validation ROC-AUC (Week 6): 0.7812


## Section 3: Leakage Audit and Failure Analysis
* **Leakage Diagnostic:** A random split leaks spatial features because survey respondents within the same Primary Sampling Unit (`hv001`) share identical regional, environmental, and infrastructure profiles. Grouping by cluster removes this cheat metric.
* **Model Failures:** The drop in ROC-AUC from 0.8316 to 0.7812 proves that the model struggles slightly more when generalizing to entirely new geographic clusters where baseline baseline conditions differ.

## Section 4: Public-Safe Claim Rewrites
* **Before (Over-hyped):** "The machine learning model accurately predicts regional household stunting risk with over 83% accuracy across Pakistan."
* **After (Safe, Directional Language):** "Using structural cluster validation, we observed a directional relationship between localized regional variables and household stunting risk, yielding a group-validated ROC-AUC of 0.7812 to serve as decision-support for public health resource allocation."

## Section 5: Self-Check
* [x] Names two paper findings and methodology questions.
* [x] Re-runs model under a grouped split with a before/after comparison.
* [x] Includes leakage audit and explicitly tracks the drop in validation metrics.
* [x] Rewrites claims using safe, defensive engineering language.